# Day 1: Hypothesis tests behind stepwise selection

**Goal:** by the end you can explain, and compute by hand, what a p-value, a Wald test, a likelihood-ratio test and a score test are, and why SAS stepwise uses the ones it does.

**Rules**
- Write every `TODO` yourself. No AI writing the code. Docs (pandas, statsmodels, scipy) are fine; that's what you'd have in the test.
- Each exercise ends with a check cell. If it passes, move on. If it fails, fix your code, don't touch the check.
- The markdown questions matter as much as the code. Answer them **out loud**, as if to the Head of Model Development. If you can't, that's the gap.
- Solutions are in the other notebook. Open it only after you've passed a check, or been stuck for 20+ minutes.

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt

## Setup: synthetic loan book (given, don't edit)

Five candidate variables. Four genuinely drive default. `noise` does not: its true coefficient is exactly zero. Keep that in mind; it's your ground truth for the whole notebook.

In [11]:
TRUE_COEFS = {'const': -5.0, 'dti': 3.0, 'ltv': 1.5, 'months_in_arrears': 0.9, 'utilisation': 1.2, 'noise': 0.0}
FEATURES = ['dti', 'ltv', 'months_in_arrears', 'utilisation', 'noise']

def make_loans(n, seed):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        'dti': rng.normal(0.35, 0.12, n).clip(0, 1),
        'ltv': rng.normal(0.75, 0.15, n).clip(0.1, 1.2),
        'months_in_arrears': rng.poisson(0.3, n),
        'utilisation': rng.beta(2, 5, n),
        'noise': rng.normal(0, 1, n),
    })
    lin = TRUE_COEFS['const'] + sum(TRUE_COEFS[f] * df[f] for f in FEATURES)
    df['true_pd'] = 1 / (1 + np.exp(-lin))
    df['default'] = rng.binomial(1, df['true_pd'])
    # Backtesting PDs with the binomial:
# If PDs are correct, the no. of defaults in a grade ~ Binomial(N, PD).
# Compare the OBSERVED default count to that distribution:
#   - analytically: binomial (or Jeffreys) test per grade -> p-value
#   - by simulation: draw default flags from the PDs many times (e.g. 10,000),
#     count defaults in each simulated book, and see where the real count falls.
#     If it's in the extreme tail (e.g. top 2.5%), the PDs look too low.
# Do it per rating grade, not just the whole book: errors in different
# grades can cancel out at book level.
# Caveat: assumes defaults are independent. In reality they're correlated
# (the economic cycle hits everyone), so the real spread is wider and the
# binomial test rejects good models too often in downturn years.
    return df

df = make_loans(5000, seed=42)
df.head()

,dti,ltv,months_in_arrears,utilisation,noise,true_pd,default
0,0.386566,0.711307,1,0.183514,0.450061,0.160687,1
1,0.225202,0.705077,1,0.240490,2.073894,0.111235,0
2,0.440054,0.645501,0,0.293653,-0.354710,0.086337,0
3,0.462868,0.814272,0,0.463544,1.023104,0.137794,0
4,0.115876,0.785197,0,0.215259,0.352892,0.038558,0


## Exercise 0: pandas warm-up (10 mins)

Build `default_by_arrears`: a DataFrame indexed by `months_in_arrears`, with columns `default_rate` and `n_loans`.

*Hint: `groupby` + `agg` with named aggregations.*

In [16]:
default_by_arrears = df.groupby('months_in_arrears').agg(
    default_rate=('default', 'mean'),
    n_loans=('default', 'count'),
)
default_by_arrears

,default_rate,n_loans
months_in_arrears,,
0,0.080942,3694
1,0.179533,1114
2,0.369318,176
3,0.666667,15
4,1.000000,1


In [17]:
# CHECK
assert list(default_by_arrears.columns) == ['default_rate', 'n_loans']
assert default_by_arrears['n_loans'].sum() == len(df)
assert default_by_arrears['default_rate'].iloc[:3].is_monotonic_increasing
print('Exercise 0 passed')

Exercise 0 passed


**Say out loud:** the bucket for 3+ months in arrears has very few loans. What would you do with it before using arrears as a binned variable, and why?

## Exercise 1: fit the model (10 mins)

Fit a logistic regression of `default` on all five `FEATURES` with an intercept. Store the fitted result as `full`.

*Hint: `sm.add_constant`, `sm.Logit(...).fit(disp=0)`.*

In [18]:
X = sm.add_constant(df[FEATURES])
full = sm.Logit(df['default'], X).fit(disp=0)
full.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                default   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4994
Method:                           MLE   Df Model:                            5
Date:                Sat, 31 Oct 2026   Pseudo R-squ.:                 0.08068
Time:                        12:49:13   Log-Likelihood:                -1640.3
converged:                       True   LL-Null:                       -1784.2
Covariance Type:            nonrobust   LLR p-value:                 4.001e-60
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -5.1198      0.308    -16.650      0.000      -5.723      -4.517
dti                   2.8863      0.390      7.408      0.000       2.123       3.650
ltv                   1.7948      0.307      5.841      0.000       1.193       2.397
months_in_arrears     0.9885      0.068     14.479      0.000       0.855       1.122
utilisation           0.8374      0.277      3.024      0.002       0.295       1.380
noise                 0.0020      0.045      0.045      0.964      -0.087       0.091
=====================================================================================
"""

In [20]:
# CHECK
assert len(full.params) == 6
assert abs(full.params['dti'] - 3.0) < 1.0
print('Exercise 1 passed')
print(pd.DataFrame({'estimate': full.params, 'true': pd.Series(TRUE_COEFS)}).round(3))

Exercise 1 passed
                   estimate  true
const                -5.120  -5.0
dti                   2.886   3.0
ltv                   1.795   1.5
months_in_arrears     0.989   0.9
utilisation           0.837   1.2
noise                 0.002   0.0


**Say out loud:** the estimates aren't exactly the true values. Why not? And what does `bse` (the standard error) measure?

## Exercise 2: the Wald test by hand (15 mins)

For each coefficient, the Wald statistic is

$$W = \left(\frac{\hat\beta}{SE(\hat\beta)}\right)^2$$

If the true coefficient is zero, $W$ follows a chi-squared distribution with 1 degree of freedom. The p-value is the probability of seeing a $W$ at least this large **if the true coefficient were zero**.

Compute `wald` (a Series) and `p_wald` (a Series) for all six parameters.

*Hint: `full.params`, `full.bse`, `stats.chi2.sf`.*

In [ ]:
# TODO
wald = ...
p_wald = ...
pd.DataFrame({'wald': wald, 'p_wald': p_wald}).round(4)

In [ ]:
# CHECK: your hand-computed p-values must match statsmodels' own
assert np.allclose(p_wald, full.pvalues)
print('Exercise 2 passed')

**Say out loud:**
1. statsmodels reports a *z*-statistic, not a chi-squared. Why did your chi-squared p-values match exactly? (Hint: what's $z^2$?)
2. What's the p-value for `noise`? Explain what it means *without* saying "the probability the variable is useless". That phrasing is wrong, and a modeller will notice.

## Exercise 3: the likelihood-ratio test (20 mins)

The LR test asks: how much worse does the model fit if I drop this variable?

$$LR = 2\,(\ell_{full} - \ell_{reduced})$$

where $\ell$ is the log-likelihood (`.llf`). Under the null, $LR \sim \chi^2_1$ when one variable is dropped.

Write a function `lr_test(data, features, drop)` that fits the full model and the model without `drop`, and returns `(lr_stat, p_value)`.

In [ ]:
# TODO
def lr_test(data, features, drop):
    ...

lr_noise = lr_test(df, FEATURES, 'noise')
lr_dti = lr_test(df, FEATURES, 'dti')
print('noise:', lr_noise)
print('dti:  ', lr_dti)

In [ ]:
# CHECK
assert lr_noise[1] > 0.05, 'noise should not be significant'
assert lr_dti[1] < 0.001, 'dti should be highly significant'
assert abs(lr_noise[0] - wald['noise']) < 1.0, 'LR and Wald should be close at this sample size'
print('Exercise 3 passed')
print(f"noise: Wald={wald['noise']:.3f}  LR={lr_noise[0]:.3f}")
print(f"dti:   Wald={wald['dti']:.3f}  LR={lr_dti[0]:.3f}")

**Say out loud:** Wald needed one model fit. LR needed two. Why might a stepwise procedure that tests 50 candidate variables at every step care about that?

## Exercise 4: what a p-value really is, by simulation (25 mins)

This is the most important exercise. `noise` truly has zero effect. So: if you regenerate the loan book many times and test `noise` each time, how often does it come out "significant" at 5%?

Write a loop that runs 500 times. Each time:
1. Make a new dataset with `make_loans(2000, seed=i)`.
2. Fit the full model.
3. Store the p-value for `noise`.

Store the results in a numpy array `null_pvals`. Then plot a histogram of them.

In [ ]:
# TODO
null_pvals = ...

# plot here


In [ ]:
# CHECK
assert len(null_pvals) == 500
false_pos = (null_pvals < 0.05).mean()
assert 0.02 < false_pos < 0.09
print(f'Exercise 4 passed. noise was "significant" in {false_pos:.1%} of datasets')

**Say out loud:**
1. What shape is the histogram, and why does it have to be that shape when the true effect is zero?
2. A stepwise procedure tries 30 candidate variables, all pure noise, each at p < 0.05. Roughly how likely is it that at least one gets in? (Work it out: $1 - 0.95^{30}$.) What does this tell you about the p-values printed for the *final* stepwise model?
3. How would you put point 2 to the Head of Model Development in one sentence, as a known limitation of stepwise?

## Exercise 5 (stretch): the score test (30 mins)

SAS stepwise uses the **score** test to decide which variable enters. Its advantage: you only fit the *smaller* model. You never fit the model with the candidate in it.

Procedure for testing whether `noise` should enter a model with the other four:
1. Fit the reduced model (no `noise`). Get its fitted probabilities $p_0$.
2. Build the full design matrix $X$ (constant + all five features).
3. Score vector: $U = X^\top (y - p_0)$
4. Information matrix: $I = X^\top W X$, where $W$ is diagonal with entries $p_0(1-p_0)$.
5. Score statistic: $S = U^\top I^{-1} U$, with $S \sim \chi^2_1$.

Compute `score_stat` and `p_score`.

*Hint: for step 4, `X.T @ (X * w[:, None])` avoids building a 5000×5000 matrix.*

In [ ]:
# TODO
score_stat = ...
p_score = ...
print(score_stat, p_score)

In [ ]:
# CHECK
assert abs(score_stat - lr_noise[0]) < 1.0
print('Exercise 5 passed')
print(f"noise -> Wald: {wald['noise']:.3f} | LR: {lr_noise[0]:.3f} | Score: {score_stat:.3f}")

**Say out loud:**
1. Three tests, three slightly different numbers, same conclusion. Why do they agree in large samples?
2. Why does SAS use score for **entry** and Wald for **removal**? Think about which model is already fitted at each point.
3. Migration angle: if the Python pipeline used LR instead of score for entry, could it select a different set of variables than SAS? When?

---
**Done?** Paste me your answers to the "say out loud" questions in your own words, rough is fine, and I'll stress-test them. Day 3 builds SAS-style stepwise from these pieces.